# Session 6 — Storage, Reproducibility & Pipeline
**Phase 3+4 Combined | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

A backtest you can't reproduce is a liability, not an asset.

Six months from now:
- Did you use adjusted or unadjusted prices?
- What version of the data was current on that date?
- Did you forward-fill or drop gaps?
- What was the Python/pandas/sklearn version?

If you can't answer these questions exactly, your results are unverifiable.

> 💡 Reproducibility is what separates a research notebook from production infrastructure. This session covers Parquet, environment management, and deterministic pipelines — the tools that make "runs identically anywhere" actually true.


---
## 1️⃣ Storage Formats — When to Use What

| Format | Speed | Size | Type-safe | SQL queries | Use when |
|--------|-------|------|-----------|-------------|----------|
| CSV | Slow | Large | ❌ No | ❌ No | Sharing with non-Python users |
| Parquet | **Fast** | **Small** | ✅ Yes | ✅ (via DuckDB) | Everything else in quant |
| HDF5 | Fast | Medium | Partial | ❌ No | Large numeric arrays |
| SQLite/SQL | Medium | Medium | ✅ Yes | ✅ Yes | Relational data, flexible queries |
| Feather | Fast | Small | ✅ Yes | ❌ No | R↔Python interop |

**Why Parquet wins for quant finance:**
- Columnar format: read only the columns you need (read `SPY` prices without loading `QQQ`)
- Compressed by default: 5-10x smaller than CSV
- Preserves dtypes: DatetimeIndex, float64, category all survive the round-trip
- Industry standard: Spark, DuckDB, pandas, Polars all read it natively


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import os, time, json
import hashlib
import warnings
warnings.filterwarnings('ignore')

# Build a sample dataset
tickers = ['SPY', 'QQQ', 'IWM', 'GLD', 'TLT']
prices = yf.download(tickers, start='2020-01-01', auto_adjust=True)['Close']
prices.index.name = 'Date'

# Create output directory
os.makedirs('data/benchmark', exist_ok=True)

# Benchmark: CSV vs Parquet

# Save CSV
t0 = time.time()
prices.to_csv('data/benchmark/prices.csv')
csv_write = time.time() - t0
csv_size  = os.path.getsize('data/benchmark/prices.csv') / 1024

# Save Parquet
t0 = time.time()
prices.to_parquet('data/benchmark/prices.parquet')
pq_write = time.time() - t0
pq_size  = os.path.getsize('data/benchmark/prices.parquet') / 1024

# Read CSV
t0 = time.time()
_ = pd.read_csv('data/benchmark/prices.csv', index_col='Date', parse_dates=True)
csv_read = time.time() - t0

# Read Parquet
t0 = time.time()
_ = pd.read_parquet('data/benchmark/prices.parquet')
pq_read = time.time() - t0

print(f"{'Format':<12} {'Size (KB)':>10} {'Write (ms)':>12} {'Read (ms)':>10}")
print("-" * 46)
print(f"{'CSV':<12} {csv_size:>10.1f} {csv_write*1000:>12.1f} {csv_read*1000:>10.1f}")
print(f"{'Parquet':<12} {pq_size:>10.1f} {pq_write*1000:>12.1f} {pq_read*1000:>10.1f}")
print(f"\nParquet is {csv_size/pq_size:.1f}x smaller and {csv_read/pq_read:.1f}x faster to read.")


In [ ]:
# Parquet: selective column reads (key advantage)
# In CSV you load everything. In Parquet you read only what you need.

t0 = time.time()
spy_only = pd.read_parquet('data/benchmark/prices.parquet', columns=['SPY'])
selective_read = time.time() - t0

t0 = time.time()
all_data = pd.read_parquet('data/benchmark/prices.parquet')
full_read = time.time() - t0

print(f"Read 1 column:   {selective_read*1000:.1f}ms")
print(f"Read 5 columns:  {full_read*1000:.1f}ms")
print(f"\nFor a 500-ticker universe, selective reads are critical for performance.")
print()

# Verify dtype preservation (CSV loses these)
csv_loaded = pd.read_csv('data/benchmark/prices.csv', index_col='Date', parse_dates=True)
pq_loaded  = pd.read_parquet('data/benchmark/prices.parquet')

print("CSV index dtype:     ", csv_loaded.index.dtype)
print("Parquet index dtype: ", pq_loaded.index.dtype)
print("CSV value dtype:     ", csv_loaded.dtypes.iloc[0])
print("Parquet value dtype: ", pq_loaded.dtypes.iloc[0])


---
## 2️⃣ Reproducibility — The Full Stack

True reproducibility requires versioning at every layer:

| Layer | Tool | What to capture |
|-------|------|-----------------|
| Data | Metadata file | Source, fetch timestamp, date range, cleaning decisions |
| Code | Git commit hash | Exact code version that produced the output |
| Environment | `requirements.txt` / `pip freeze` | Exact library versions |
| Random state | `random_state=42` everywhere | Any stochastic operation |
| Pipeline | Idempotent design | Same input → same output, always |


In [ ]:
import subprocess
import platform

def get_git_hash() -> str:
    # Get current git commit hash for provenance tracking
    try:
        result = subprocess.run(
            ['git', 'rev-parse', 'HEAD'],
            capture_output=True, text=True
        )
        return result.stdout.strip()[:8] if result.returncode == 0 else 'not-a-git-repo'
    except FileNotFoundError:
        return 'git-not-found'

def get_file_hash(filepath: str) -> str:
    # Compute SHA256 hash of a file for determinism verification
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()[:16]

def build_metadata(df: pd.DataFrame, source: str, cleaning_decisions: list) -> dict:
    return {
        'created_at':          pd.Timestamp.now().isoformat(),
        'git_commit':          get_git_hash(),
        'python_version':      platform.python_version(),
        'pandas_version':      pd.__version__,
        'numpy_version':       np.__version__,
        'source':              source,
        'tickers':             list(df.columns),
        'date_range':          [str(df.index.min().date()), str(df.index.max().date())],
        'n_rows':              len(df),
        'n_tickers':           len(df.columns),
        'cleaning_decisions':  cleaning_decisions,
    }

# Example usage
cleaning_decisions = [
    "Prices forward-filled for gaps <= 3 business days",
    "Gaps > 3 business days left as NaN and logged",
    "auto_adjust=True (split and dividend adjusted)",
]

metadata = build_metadata(prices, source='yfinance', cleaning_decisions=cleaning_decisions)

# Save alongside data
os.makedirs('data', exist_ok=True)
with open('data/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved:")
print(json.dumps(metadata, indent=2))


In [ ]:
# Determinism test — same input must produce same output
# Run the pipeline twice, check file hashes match

def run_pipeline(tickers, start, output_path):
    # Deterministic pipeline: same inputs -> identical output file
    np.random.seed(42)  # Fix any stochastic operations

    data = yf.download(tickers, start=start, auto_adjust=True, progress=False)['Close']
    data = data.ffill()  # deterministic operation

    # Deterministic feature engineering
    log_ret = np.log(data / data.shift(1))
    features = pd.concat([
        data.add_suffix('_price'),
        log_ret.add_suffix('_ret'),
        log_ret.rolling(20).std().mul(np.sqrt(252)).add_suffix('_vol'),
    ], axis=1).dropna()

    features.to_parquet(output_path)
    return get_file_hash(output_path)

hash1 = run_pipeline(tickers, '2022-01-01', 'data/run1.parquet')
hash2 = run_pipeline(tickers, '2022-01-01', 'data/run2.parquet')

print("Determinism test:")
print(f"  Run 1 hash: {hash1}")
print(f"  Run 2 hash: {hash2}")
print(f"  {'✅ IDENTICAL — pipeline is deterministic' if hash1 == hash2 else '❌ DIFFERENT — pipeline is not deterministic'}")


---
## 3️⃣ When to Recompute vs When to Persist

| Data Type | Strategy | Reasoning |
|-----------|----------|-----------|
| Raw prices | Persist | Immutable source of truth — never recompute |
| Cleaned prices | Persist | Recompute only if cleaning rules change (log the change) |
| Features | Persist | Expensive to recompute; cache with metadata |
| Model outputs | Persist | Reproducibility for attribution |
| Diagnostics / reports | Recompute | Cheap; always want fresh reports |

**Versioning data files:**
```
data/
  prices_v1_2024-01-01.parquet   ← initial fetch
  prices_v2_2024-06-15.parquet   ← after adding new tickers
  features_v1_2024-01-01.parquet
  metadata_v1.json
```
Or use hash-based naming: `prices_abc123de.parquet` where the hash encodes the config.


---
## ✅ Self-Check Questions

1. Why is Parquet the preferred format for quant data pipelines over CSV?
   > *Columnar format (read only needed columns), compressed (5-10x smaller), type-safe (dtypes preserved), industry standard, fast for large datasets.*

2. What does "idempotent pipeline" mean?
   > *Running it twice with the same inputs produces identical outputs. Essential for debugging and reproducibility.*

3. What four things need to be versioned for full reproducibility?
   > *Data (source + fetch config), code (git hash), environment (requirements.txt / pip freeze), and any random seeds.*

4. When should you recompute vs persist intermediate results?
   > *Persist: expensive computations, raw source data, anything needed for audit. Recompute: cheap diagnostics, reports, anything derived from persisted data.*

---
## 🧪 Optional Tasks

- Benchmark CSV vs Parquet for a 500-ticker universe (download all S&P 500 ETF proxies). At what scale does the speed difference become critical?
- Break determinism intentionally (e.g., use `pd.Timestamp.now()` in feature names). Confirm hash mismatch.
- Query your Parquet file with DuckDB: `import duckdb; duckdb.query("SELECT Date, SPY FROM 'data/prices.parquet' WHERE SPY > 400")`. Notice how SQL+Parquet gives you database-like access without a database.
- Add a version field to your metadata. Write a loader that rejects data if the schema version is incompatible.
